## LLM-Judge Scoring of Policy Phrases

<a href="https://colab.research.google.com/github/crezny/DATASCI266_final_project/blob/main/LLM-Judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install neo4j


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 14.9 MB/s eta 0:00:00


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import re
from neo4j import GraphDatabase
from collections import defaultdict
import pandas as pd
import os
import re
import numpy as np
import math
try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm


from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
try:
    from google.colab import userdata
    get_secret = userdata.get
except ImportError:
    # Fallback: define a dummy get_secret for local use
    def get_secret(key):
        return None

# from dotenv import load_dotenv
# load_dotenv()


from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

def get_env(key: str, default: str = None):
    """Tries os.getenv first, then Colab userdata if available."""
    return os.getenv(key) or get_secret(key) or default

NEO4J_URI  = get_env("NEO4J_URI")   # bolt:// or neo4j://
NEO4J_USER = get_env("NEO4J_USER")
NEO4J_PASS = get_env("NEO4J_PASS")
NEO4J_DATABASE = get_env("NEO4J_DATABASE")

DB_HOST = get_env("POSTGRES_DB_HOST")
DB_PORT = int(get_env("POSTGRES_DB_PORT"))
DB_NAME = get_env("POSTGRES_DB_NAME")
DB_USER = get_env("POSTGRES_DB_USER")
DB_PASS = get_env("POSTGRES_DB_PASS")

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"



In [4]:
# Skywork Reward Model Evaluator
# ===============================
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import re

# --- Load the Skywork reward model ---
model_name = "Skywork/Skywork-Reward-V2-Llama-3.1-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32  # safe default




model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    dtype=dtype,
    device_map= device,
    num_labels=1,  # reward models output a single scalar


)

# --- Token IDs for binary "Yes"/"No" ---
YES_TOKEN_ID = tokenizer.encode("Yes", add_special_tokens=False)[0]
NO_TOKEN_ID  = tokenizer.encode("No",  add_special_tokens=False)[0]


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/389 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/993 [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/117M [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
# Message construction aligned to Skywork reward model format
# ============================================================

POLICY_PRINCIPLE = """
You are evaluating a single policy clause. The clause may have multiple subsections.
You will be provided with the applicable framework, and the controls and subcontrols
that the clause is intended to address. You may also be supplied with the organization's
context (industry, size, risk tolerance, key assets, etc.) when available.
If this context is not provided, you should evaluate the clause based solely on the framework requirements.

Evaluate how well the clause satisfies the following criteria:
1. Be precise, enforceable, and free of ambiguity.
2. Correctly align with the stated framework requirements (e.g., NIST CSF, SOC 2, ISO 27001) when referenced.
3. Reflect the organization's ontology (industry, size, risk tolerance, key assets, etc.) when such context is provided.
4. Avoid hallucinated frameworks, controls, or obligations.
5. Maintain a professional, directive tone appropriate for a formal corporate policy.

Special evaluation rules:
- **If no specific controls or subcontrols are provided**:
  Do *not* penalize the clause for lacking explicit control references.
  Instead, focus on clarity, correctness, and appropriateness for its section type.

- **If the section is an introductory or structural section**
  (for example *Purpose*, *Scope*, *Definitions*, *Roles & Responsibilities*):
  Evaluate it on clarity of intent, relevance to the policy and framework theme, and professional tone.
  Such sections are not expected to contain control implementation details or enforceable requirements.

- **Only assign low scores (<50)** when the clause is clearly off-topic,
  factually incorrect, internally contradictory, or so vague it fails to communicate intent.

Assign a numeric score from 0 to 100, where:
- 0 = completely non-compliant with the principle,
- 50 = partially meets requirements but contains material gaps or ambiguities, and
- 100 = fully compliant and exemplary in clarity, alignment, and tone.

Your goal is to output a single numeric score reflecting how well the clause adheres to these principles.
Do not include any explanations or commentary — only return the numeric score.
""".strip()

def skywork_raw_reward_for_clause(clause_text: str,
                                  subcontrol_text: str) -> float:
    """
    Returns the raw Skywork reward (unbounded scalar) for how well
    `clause_text` performs as an 'answer' under the constraint of `subcontrol_text`.
    """
    messages = [
        {
            "role": "user",
            "content": (
                "You are evaluating the next assistant message, which is a proposed policy clause. "
                "It may include multiple subsections. Use the following principle and framework context to judge it.\n\n"
                f"{POLICY_PRINCIPLE}\n\n"
                "Framework sub-control requirement:\n"
                f"{subcontrol_text}\n\n"
                "Reward higher if it is:\n"
                "- Precise and enforceable (no vague language)\n"
                "- Directly aligned with the sub-control scope and obligations\n"
                "- Complete on key points (who, what, when, how)\n"
                "- Free of contradictions or hallucinated requirements\n"
                "- Professional and directive in tone\n"
                "Penalize if it is vague, off-scope, incorrect, or incomplete."
            ),
        },
        {
            # This is the thing being judged
            "role": "assistant",
            "content": clause_text.strip(),
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    # Model card note: avoid duplicate BOS if present.
    if tokenizer.bos_token and text.startswith(tokenizer.bos_token):
        text = text[len(tokenizer.bos_token):]

    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)

    with torch.no_grad():
        out = model(**inputs)
        raw = out.logits[0][0].item()

    return float(raw)


In [6]:
def skywork_clause_framework_score(clause_text: str,
                                   subcontrol_text: str) -> float:
    """
    Returns a 0-100 adherence score using Skywork RM as backbone.
    """
    raw = skywork_raw_reward_for_clause(clause_text, subcontrol_text)
    # Logistic squashing: maps (-inf, +inf) -> (0, 1)
    prob = 1.0 / (1.0 + math.exp(-raw))

    # Scale to 0-100
    return float(prob * 100.0)

In [7]:
# Neo4j Framework Context Extraction
# ============================================================

CYPHER_POLICY_FRAMEWORK_MAP = """
MATCH (f:Framework)
WHERE f.name = $framework_name
MATCH (f)-[:HAS_CONTROL_AREA]->(ca:ControlArea)
MATCH (ca)-[:HAS_SUBCONTROL]->(sc:SubControl)
MATCH (sc)-[:REQUIRES_EVIDENCE]->(er:EvidenceRequirement)
MATCH (p:Policy)-[:SATISFIES_REQUIREMENT]->(er)
WHERE p.title = $policy_name
RETURN DISTINCT
  f.name   AS framework_name,
  p.title  AS policy_title,
  ca.name  AS control_name,
  sc.title AS subcontrol_name,
  sc.code    AS subcontrol_id
ORDER BY control_name, subcontrol_name
"""

def get_framework_context_for_policy(policy_name: str,
                                     framework_name: str = "NIST CSF") -> dict:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
    try:
        with driver.session(database=NEO4J_DATABASE) as session:
            rows = session.run(
                CYPHER_POLICY_FRAMEWORK_MAP,
                framework_name=framework_name,
                policy_name=policy_name,
            ).data()
    finally:
        driver.close()

    # Group by control area
    by_control = defaultdict(list)
    for r in rows:
        by_control[r["control_name"]].append({
            "id": r.get("subcontrol_id"),
            "name": r["subcontrol_name"],
        })

    return {
        "framework": framework_name,
        "policy": policy_name,
        "controls": [
            {
                "control_area": control,
                "subcontrols": subs,
            }
            for control, subs in by_control.items()
        ]
    }


In [8]:
def render_framework_context(framework_data: dict) -> str:
    """
    Turn your framework-control mapping into a readable context block
    for the evaluation prompt.
    """
    lines = []
    print(framework_data)
    lines.append(f"Framework: {framework_data['framework']}")
    lines.append(f"Policy: {framework_data['policy']}")
    lines.append("")

    for ctrl in framework_data.get("controls", []):
        lines.append(f"Control Area: {ctrl['control_area']}")
        for sc in ctrl.get('subcontrols', []):
            lines.append(f"  - {sc['id']}: {sc['name']}")
        lines.append("")  # blank line between control areas

    return "\n".join(lines).strip()

In [9]:
def get_section_text(df, policy_id: str, section: str, include_children=True) -> str:
    # Filter for this policy
    pdf = df[df['policy_id'] == policy_id].copy()
    if pdf.empty:
        return f"[No content found for policy {policy_id}]"

    # Choose filtering style: exact match or include children (prefix)
    if include_children:
        section_df = pdf[pdf['clause_section_number'].str.startswith(section)]
    else:
        section_df = pdf[pdf['clause_section_number'] == section]

    if section_df.empty:
        return f"[No section {section} found for policy {policy_id}]"

    output_lines = []
    last_section = None

    for _, row in section_df.iterrows():
        sec_num = row['clause_section_number']
        title = row.get('section_title', '')
        text = str(row.get('clause_text', '')).strip()

        # Print header at first appearance of each section/subsection
        if sec_num != last_section:
            header = f"{sec_num} {title}".strip()
            output_lines.append(header)
            last_section = sec_num

        # Add the body text
        if text:
            output_lines.append(text)

    return "\n".join(output_lines)


In [17]:
def apply_judge(
    df,
    prompt_col: str='clause_full_text',
    framework_col: str='source_framework',
    policy_name_col: str="policy_title",
    section_num_col: str="clause_section_number",
    section_title_col: str="section_title",
    output_col: str="output"
    ):

    scores = []
    pbar = tqdm(
        total=len(df),
        desc="Evaluating policy clauses",
        unit="clause",
        leave=True
    )

    for index, row in df.iterrows():
        policy_name = row.get(policy_name_col)
        section_num = row.get(section_num_col, "")
        section_title = row.get(section_title_col, "")
        framework = row.get(framework_col)
        clause_text = row.get(prompt_col)

        # Update progress bar message dynamically
        pbar.set_postfix({
            "Policy": str(policy_name)[:25],
            "Section": str(section_num),
            "Framework": str(framework)
        })

        section_context = f"Policy: {policy_name}\nSection {section_num}: {section_title}".strip()
        clause_block = f"{section_context}\n\nClause to evaluate:\n{clause_text.strip()}"

        # # Build inputs and run scoring
        framework_data = get_framework_context_for_policy(policy_name, framework)
        score = skywork_clause_framework_score(clause_block, framework_data)
        scores.append(score)
        pbar.update(1)

    pbar.close()

    df[output_col] = scores
    return df


In [18]:
def add_framework_context_to_df(
    df,
    framework_col: str='source_framework',
    framework_context_col: str='framework_context'
):
    framework_contexts = []
    pbar = tqdm(
        total=len(df),
        desc="Adding framework context",
        unit="clause",
        leave=True
    )

    for index, row in df.iterrows():
        policy_name = row.get('policy_title')
        framework = row.get(framework_col)

        # Update progress bar message dynamically
        pbar.set_postfix({
            "Policy": str(policy_name)[:25],
            "Framework": str(framework)
        })


        framework_data = get_framework_context_for_policy(policy_name, framework)
        context_text = render_framework_context(framework_data)
        framework_contexts.append(context_text)
        pbar.update(1)

    pbar.close()

    df[framework_context_col] = framework_contexts
    return df

In [12]:
row = compact_df.iloc[1]
policy_name = row.get('policy_title')
section_num = row.get('clause_section_number', "")
section_title = row.get('section_title', "")
clause_text = row.get('section_text')
framework = row.get('source_framework')

section_context = f"Policy: {policy_name}\nSection {section_num}: {section_title}".strip()
clause_block = f"{section_context}\n\nClause to evaluate:\n{clause_text.strip()}"
print(section_context)
print("------")
print(clause_block)
print("======")
framework_data = get_framework_context_for_policy(policy_name, framework)
print(framework_data)

NameError: name 'compact_df' is not defined

In [11]:
connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url, echo=False)
df = pd.read_sql('SELECT * FROM policy_lines;', engine)
df.head()


,row_number,policy_id,policy_title,source_framework,policy_origin,section_id,section_title,is_heading,clause_id,clause_text,clause_full_text,clause_section_number,company_type,company_id
0,0,CL_0001-AUP-v2,Acceptable Use Policy,NIST CSF,client,CL_0001-AUP-v2:S1,PURPOSE,False,CL_0001-AUP-v2:1.context.523a34,The purpose of this policy is to outline the a...,The purpose of this policy is to outline the a...,1,Client,CL_0001
1,1,CL_0001-AUP-v2,Acceptable Use Policy,NIST CSF,client,CL_0001-AUP-v2:S2,SCOPE,False,CL_0001-AUP-v2:2.context.5af3cd,This policy applies to all City personnel and ...,This policy applies to all City personnel and ...,2,Client,CL_0001
2,2,CL_0001-AUP-v2,Acceptable Use Policy,NIST CSF,client,CL_0001-AUP-v2:S3.1,General Use and Ownership,False,CL_0001-AUP-v2:3.1.context.bf694e,[COMPANY_NAME] information stored on electroni...,[COMPANY_NAME] information stored on electroni...,3.1,Client,CL_0001
3,3,CL_0001-AUP-v2,Acceptable Use Policy,NIST CSF,client,CL_0001-AUP-v2:S3.1,General Use and Ownership,False,CL_0001-AUP-v2:3.1.context.efab41,All personnels and contractors have a responsi...,All personnels and contractors have a responsi...,3.1,Client,CL_0001
4,4,CL_0001-AUP-v2,Acceptable Use Policy,NIST CSF,client,CL_0001-AUP-v2:S3.1,General Use and Ownership,False,CL_0001-AUP-v2:3.1.context.5e59b4,"All personnels and contractors may access, use...","All personnels and contractors may access, use...",3.1,Client,CL_0001


In [13]:
# df_test = df[df['source_framework'] == 'NIST CSF']

# df_test = df_test.sample(n = 100, random_state=42 )
df_test = df
df_test = df_test.drop_duplicates(subset=['policy_id', 'clause_section_number'])
df_test = df_test.reset_index(drop=True)


compact_df = df_test[['policy_id','policy_title','source_framework','policy_origin','section_id','section_title','clause_section_number','company_type','company_id']]
compact_df = compact_df.drop_duplicates()
compact_df['section_text'] = compact_df[['policy_id', 'clause_section_number']].apply(lambda row: get_section_text(df, row['policy_id'], row['clause_section_number']),axis =1)


judged_df = apply_judge(compact_df,
    prompt_col='section_text',
    framework_col='source_framework',
    policy_name_col="policy_title",
    section_num_col="clause_section_number",
    section_title_col="section_title",
    output_col="judged_score"
)



Evaluating policy clauses:   0%|          | 0/1515 [00:00<?, ?clause/s]

In [26]:
# judged_df
# WHAT I NEED
# policy_id	model_name	output	time_taken	token_count	policy_title	section_number	variant	model_type	execution_date
import datetime
from datetime import datetime
ddf = judged_df.copy()
ddf['model_name'] = 'SME'
ddf.rename(columns={'section_text': 'output'}, inplace=True)
ddf['time_taken'] = 0
ddf['token_count'] = 0
ddf['policy_title'] = ddf['policy_title'].str.replace("'", "''")
ddf.rename(columns={'clause_section_number': 'section_number'}, inplace=True)
ddf['variant'] = 'default'
ddf['model_type'] = 'SME'
ddf['execution_date'] = datetime.now()


In [28]:
ddf = ddf[['policy_id','model_name','output','time_taken','token_count','policy_title','section_number','variant','model_type','execution_date','judged_score']].copy()

In [30]:
# ddf.head()
ddf.to_sql('judge_results', engine, if_exists='replace', index=False)

515

In [12]:

model_results_df = pd.read_sql(
    """
    SELECT model_results.* , d.source_framework, d.section_title
    FROM model_results
    JOIN (
    SELECT DISTINCT policy_id, section_id, section_title, source_framework FROM policy_lines
    ) AS D
    ON D.policy_id = model_results.policy_id and D.section_id = model_results.policy_id || ':S' || section_number

      """, engine)
model_results_df.rename(columns={'output': 'section_text'}, inplace=True)

model_results_df.head()

,policy_id,model_name,section_text,time_taken,token_count,policy_title,section_number,variant,model_type,execution_date,source_framework,section_title
0,CL_0001-ASMP-v1.0,Gemma-2-9B,This policy establishes the framework for the ...,13.18,153,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE
1,CL_0001-ASMP-v1.0,SmolLM3,The purpose of this Asset Management Policy is...,6.25,161,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE
2,CL_0001-ASMP-v1.0,Mistral-7B-Instruct-v0.3,This Asset Management Policy is established to...,15.94,256,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE
3,CL_0001-ASMP-v1.0,Llama-3.2-3B-Instruct,The purpose of this Asset Management Policy is...,6.95,196,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE
4,CL_0001-ASMP-v1.0,Llama-3.2-3B-Instruct,3.10. Telecommunications and Mobile Devices\nA...,7.23,207,Asset Management Policy,3.10,None,V1,2025-11-12 01:04:55.129793,NIST CSF,Telecommunications and Mobile Devices


In [13]:
add_framework_context_to_df(model_results_df, framework_col='source_framework', framework_context_col='framework_context')

Adding framework context:   0%|          | 0/1731 [00:00<?, ?clause/s]

{'framework': 'NIST CSF', 'policy': 'Asset Management Policy', 'controls': [{'control_area': 'Asset Management (ID.AM)', 'subcontrols': [{'id': 'ID.AM-1', 'name': 'Device and System Management (ID.AM-1)'}, {'id': 'ID.AM-4', 'name': 'External Information Systems (ID.AM-4)'}, {'id': 'ID.AM-5', 'name': 'Resources Classification (ID.AM-5)'}, {'id': 'ID.AM-2', 'name': 'Software and Applications Management (ID.AM-2)'}]}, {'control_area': 'Data Security (PR.DS)', 'subcontrols': [{'id': 'PR.DS-3', 'name': 'Asset Management (PR.DS-3)'}]}, {'control_area': 'Information Protection Processes and Procedures (PR.IP)', 'subcontrols': [{'id': 'PR.IP-5', 'name': 'Physical Operating Environment (PR.IP-5)'}]}]}
{'framework': 'NIST CSF', 'policy': 'Asset Management Policy', 'controls': [{'control_area': 'Asset Management (ID.AM)', 'subcontrols': [{'id': 'ID.AM-1', 'name': 'Device and System Management (ID.AM-1)'}, {'id': 'ID.AM-4', 'name': 'External Information Systems (ID.AM-4)'}, {'id': 'ID.AM-5', 'name

,policy_id,model_name,section_text,time_taken,token_count,policy_title,section_number,variant,model_type,execution_date,source_framework,section_title,framework_context
0,CL_0001-ASMP-v1.0,Gemma-2-9B,This policy establishes the framework for the ...,13.18,153,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...
1,CL_0001-ASMP-v1.0,SmolLM3,The purpose of this Asset Management Policy is...,6.25,161,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...
2,CL_0001-ASMP-v1.0,Mistral-7B-Instruct-v0.3,This Asset Management Policy is established to...,15.94,256,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...
3,CL_0001-ASMP-v1.0,Llama-3.2-3B-Instruct,The purpose of this Asset Management Policy is...,6.95,196,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...
4,CL_0001-ASMP-v1.0,Llama-3.2-3B-Instruct,3.10. Telecommunications and Mobile Devices\nA...,7.23,207,Asset Management Policy,3.10,None,V1,2025-11-12 01:04:55.129793,NIST CSF,Telecommunications and Mobile Devices,Framework: NIST CSF\nPolicy: Asset Management ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1726,SOC_2-LMP-v1.0,Llama-3.2-3B-Instruct,3.6. Application Log Elements\nApplication log...,3.36,93,Logging and Monitoring Policy,3.6,None,V1,2025-11-12 01:04:55.129793,SOC2,Application Log Elements,Framework: SOC2\nPolicy: Logging and Monitorin...
1727,SOC_2-LMP-v1.0,Llama-3.2-3B-Instruct,Formatting and Storage Requirements\n\nAll dat...,8.99,257,Logging and Monitoring Policy,3.8,A,baseline,2025-11-11 23:49:33.577771,SOC2,Formatting and Storage,Framework: SOC2\nPolicy: Logging and Monitorin...
1728,SOC_2-LMP-v1.0,SmolLM3,The purpose of this section is to outline the ...,6.79,175,Logging and Monitoring Policy,3.8,A,baseline,2025-11-11 23:49:33.577771,SOC2,Formatting and Storage,Framework: SOC2\nPolicy: Logging and Monitorin...
1729,SOC_2-LMP-v1.0,Mistral-7B-Instruct-v0.3,Section 3.8 - Formatting and Storage\n\nThe pu...,15.90,256,Logging and Monitoring Policy,3.8,A,baseline,2025-11-11 23:49:33.577771,SOC2,Formatting and Storage,Framework: SOC2\nPolicy: Logging and Monitorin...


In [15]:
model_results_df

,policy_id,model_name,section_text,time_taken,token_count,policy_title,section_number,variant,model_type,execution_date,source_framework,section_title,framework_context
0,CL_0001-ASMP-v1.0,Gemma-2-9B,This policy establishes the framework for the ...,13.18,153,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...
1,CL_0001-ASMP-v1.0,SmolLM3,The purpose of this Asset Management Policy is...,6.25,161,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...
2,CL_0001-ASMP-v1.0,Mistral-7B-Instruct-v0.3,This Asset Management Policy is established to...,15.94,256,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...
3,CL_0001-ASMP-v1.0,Llama-3.2-3B-Instruct,The purpose of this Asset Management Policy is...,6.95,196,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...
4,CL_0001-ASMP-v1.0,Llama-3.2-3B-Instruct,3.10. Telecommunications and Mobile Devices\nA...,7.23,207,Asset Management Policy,3.10,None,V1,2025-11-12 01:04:55.129793,NIST CSF,Telecommunications and Mobile Devices,Framework: NIST CSF\nPolicy: Asset Management ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1726,SOC_2-LMP-v1.0,Llama-3.2-3B-Instruct,3.6. Application Log Elements\nApplication log...,3.36,93,Logging and Monitoring Policy,3.6,None,V1,2025-11-12 01:04:55.129793,SOC2,Application Log Elements,Framework: SOC2\nPolicy: Logging and Monitorin...
1727,SOC_2-LMP-v1.0,Llama-3.2-3B-Instruct,Formatting and Storage Requirements\n\nAll dat...,8.99,257,Logging and Monitoring Policy,3.8,A,baseline,2025-11-11 23:49:33.577771,SOC2,Formatting and Storage,Framework: SOC2\nPolicy: Logging and Monitorin...
1728,SOC_2-LMP-v1.0,SmolLM3,The purpose of this section is to outline the ...,6.79,175,Logging and Monitoring Policy,3.8,A,baseline,2025-11-11 23:49:33.577771,SOC2,Formatting and Storage,Framework: SOC2\nPolicy: Logging and Monitorin...
1729,SOC_2-LMP-v1.0,Mistral-7B-Instruct-v0.3,Section 3.8 - Formatting and Storage\n\nThe pu...,15.90,256,Logging and Monitoring Policy,3.8,A,baseline,2025-11-11 23:49:33.577771,SOC2,Formatting and Storage,Framework: SOC2\nPolicy: Logging and Monitorin...


In [19]:
judged_df = apply_judge(model_results_df,
    prompt_col='section_text',
    framework_col='source_framework',
    policy_name_col="policy_title",
    section_num_col="section_number",
    section_title_col="section_title",
    output_col="judged_score"
)




Evaluating policy clauses:   0%|          | 0/1731 [00:00<?, ?clause/s]

In [22]:
judged_df

,policy_id,model_name,section_text,time_taken,token_count,policy_title,section_number,variant,model_type,execution_date,source_framework,section_title,framework_context,judged_score
0,CL_0001-ASMP-v1.0,Gemma-2-9B,This policy establishes the framework for the ...,13.18,153,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...,98.967185
1,CL_0001-ASMP-v1.0,SmolLM3,The purpose of this Asset Management Policy is...,6.25,161,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...,97.023991
2,CL_0001-ASMP-v1.0,Mistral-7B-Instruct-v0.3,This Asset Management Policy is established to...,15.94,256,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...,61.091782
3,CL_0001-ASMP-v1.0,Llama-3.2-3B-Instruct,The purpose of this Asset Management Policy is...,6.95,196,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...,97.805175
4,CL_0001-ASMP-v1.0,Llama-3.2-3B-Instruct,3.10. Telecommunications and Mobile Devices\nA...,7.23,207,Asset Management Policy,3.10,None,V1,2025-11-12 01:04:55.129793,NIST CSF,Telecommunications and Mobile Devices,Framework: NIST CSF\nPolicy: Asset Management ...,0.053578
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1726,SOC_2-LMP-v1.0,Llama-3.2-3B-Instruct,3.6. Application Log Elements\nApplication log...,3.36,93,Logging and Monitoring Policy,3.6,None,V1,2025-11-12 01:04:55.129793,SOC2,Application Log Elements,Framework: SOC2\nPolicy: Logging and Monitorin...,99.980890
1727,SOC_2-LMP-v1.0,Llama-3.2-3B-Instruct,Formatting and Storage Requirements\n\nAll dat...,8.99,257,Logging and Monitoring Policy,3.8,A,baseline,2025-11-11 23:49:33.577771,SOC2,Formatting and Storage,Framework: SOC2\nPolicy: Logging and Monitorin...,78.266248
1728,SOC_2-LMP-v1.0,SmolLM3,The purpose of this section is to outline the ...,6.79,175,Logging and Monitoring Policy,3.8,A,baseline,2025-11-11 23:49:33.577771,SOC2,Formatting and Storage,Framework: SOC2\nPolicy: Logging and Monitorin...,97.771381
1729,SOC_2-LMP-v1.0,Mistral-7B-Instruct-v0.3,Section 3.8 - Formatting and Storage\n\nThe pu...,15.90,256,Logging and Monitoring Policy,3.8,A,baseline,2025-11-11 23:49:33.577771,SOC2,Formatting and Storage,Framework: SOC2\nPolicy: Logging and Monitorin...,86.246753


In [28]:
ddf = judged_df.copy()
ddf.rename(columns={'section_text': 'output'}, inplace=True)

ddf = ddf[['policy_id','model_name','output','time_taken','token_count','policy_title','section_number','variant','model_type','execution_date','judged_score']]
ddf.to_sql('judge_results', engine, if_exists='append', index=False)

731

In [24]:
ddf

,policy_id,model_name,output,time_taken,token_count,policy_title,section_number,variant,model_type,execution_date,source_framework,section_title,framework_context,judged_score
0,CL_0001-ASMP-v1.0,Gemma-2-9B,This policy establishes the framework for the ...,13.18,153,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...,98.967185
1,CL_0001-ASMP-v1.0,SmolLM3,The purpose of this Asset Management Policy is...,6.25,161,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...,97.023991
2,CL_0001-ASMP-v1.0,Mistral-7B-Instruct-v0.3,This Asset Management Policy is established to...,15.94,256,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...,61.091782
3,CL_0001-ASMP-v1.0,Llama-3.2-3B-Instruct,The purpose of this Asset Management Policy is...,6.95,196,Asset Management Policy,1,A,baseline,2025-11-11 23:49:33.577771,NIST CSF,PURPOSE,Framework: NIST CSF\nPolicy: Asset Management ...,97.805175
4,CL_0001-ASMP-v1.0,Llama-3.2-3B-Instruct,3.10. Telecommunications and Mobile Devices\nA...,7.23,207,Asset Management Policy,3.10,None,V1,2025-11-12 01:04:55.129793,NIST CSF,Telecommunications and Mobile Devices,Framework: NIST CSF\nPolicy: Asset Management ...,0.053578
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1726,SOC_2-LMP-v1.0,Llama-3.2-3B-Instruct,3.6. Application Log Elements\nApplication log...,3.36,93,Logging and Monitoring Policy,3.6,None,V1,2025-11-12 01:04:55.129793,SOC2,Application Log Elements,Framework: SOC2\nPolicy: Logging and Monitorin...,99.980890
1727,SOC_2-LMP-v1.0,Llama-3.2-3B-Instruct,Formatting and Storage Requirements\n\nAll dat...,8.99,257,Logging and Monitoring Policy,3.8,A,baseline,2025-11-11 23:49:33.577771,SOC2,Formatting and Storage,Framework: SOC2\nPolicy: Logging and Monitorin...,78.266248
1728,SOC_2-LMP-v1.0,SmolLM3,The purpose of this section is to outline the ...,6.79,175,Logging and Monitoring Policy,3.8,A,baseline,2025-11-11 23:49:33.577771,SOC2,Formatting and Storage,Framework: SOC2\nPolicy: Logging and Monitorin...,97.771381
1729,SOC_2-LMP-v1.0,Mistral-7B-Instruct-v0.3,Section 3.8 - Formatting and Storage\n\nThe pu...,15.90,256,Logging and Monitoring Policy,3.8,A,baseline,2025-11-11 23:49:33.577771,SOC2,Formatting and Storage,Framework: SOC2\nPolicy: Logging and Monitorin...,86.246753
